In [ ]:
import pandas as pd
import numpy as np
import sys 
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.nn.parameter import Parameter
import time 

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

import math
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../..'))

from GNNs.GCN import GCN
#from lstm_dataset import TimeSeriesDataset
from model_utils.plots import plot_results

In [2]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: grid_search_plots
Removed directory: inference_logs


In [3]:
DATA_PATH = '../../dataset/independent_items.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

NUM_ITEMS = 100
df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../../dataset/independent_items.feather...


In [4]:
df["item_id"].unique()

array([    27, 606214,  20253, 606230, 606279, 270075, 606257,  48375,
          779, 606290, 606402, 606415, 945964,    903,  20082, 606300,
       606365, 606383, 959943, 124581,  47364,  21161, 962469, 288716,
       587317,  20605,  48373, 961844,  48371,  48372,  20965, 962444,
       937490, 237588, 132359, 132363, 132357,   1260, 628129, 241839,
       132356,  97739, 640017,   1325, 640028,   1327,  16961, 936601,
       936693, 613934, 613935,  19615, 129834,    904,  53667, 945892,
        18607,  18605,    980, 256207, 538923, 113775,  26636, 501490,
       501469, 501471,    314, 545598, 983754, 501711,  26862,  26792,
          278, 536433, 501132,    347,    348,    360,  26164,  26220,
       545621,    324, 545604,    342,  26304,  32239,  32180, 113887,
       535948,  29629, 988016, 530684, 505414,    156,  30753,  26924,
          260, 113632, 536321,  26910])

In [5]:
import holidays

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)
ITEM_COL = 'item_id'

# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 154
forecast_horizon = 152

# -----------------------------------------------------------------------------
# SORT
# -----------------------------------------------------------------------------
df = df.sort_values([DATE_COL, "item_id", "store_id"]).reset_index(drop=True)

# -----------------------------------------------------------------------------
# BASIC CALENDAR PARTS
# -----------------------------------------------------------------------------
df["day_of_week"]   = df[DATE_COL].dt.dayofweek.astype(int)         # 0=Mon
df["day_of_month"]  = df[DATE_COL].dt.day.astype(int)
df["month"]         = df[DATE_COL].dt.month.astype(int)
df["moy"]           = (df["month"] - 1).astype(int)
df["quarter"]       = df[DATE_COL].dt.quarter.astype(int)
df["doy"]           = (df[DATE_COL].dt.dayofyear - 1).astype(int)
df["week_of_year"]  = df[DATE_COL].dt.isocalendar().week.astype(int)
df["year"]          = df[DATE_COL].dt.year.astype(int)

# weekend
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)


# month / quarter boundaries
df["is_month_start"]   = df[DATE_COL].dt.is_month_start.astype(int)
df["is_month_end"]     = df[DATE_COL].dt.is_month_end.astype(int)
df["is_quarter_start"] = df[DATE_COL].dt.is_quarter_start.astype(int)
df["is_quarter_end"]   = df[DATE_COL].dt.is_quarter_end.astype(int)

# optional: week of month
df["week_of_month"] = ((df["day_of_month"] - 1) // 7 + 1).astype(int)

# -----------------------------------------------------------------------------
# CYCLICAL ENCODINGS
# -----------------------------------------------------------------------------
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)

# -----------------------------------------------------------------------------
# US HOLIDAYS
# -----------------------------------------------------------------------------
us_holidays = holidays.US()
holiday_dates = pd.to_datetime(sorted(us_holidays.keys()))

df["is_holiday"] = df[DATE_COL].isin(holiday_dates).astype(int)

# named holidays
df["is_christmas"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)
).astype(int)

df["is_thanksgiving"] = df[DATE_COL].apply(
    lambda x: 1 if us_holidays.get(x) == "Thanksgiving Day" else 0
)
# Black Friday (very useful for retail)
thanksgiving_dates = pd.to_datetime(
    [d for d, name in us_holidays.items() if name == "Thanksgiving Day"]
)
black_friday_dates = thanksgiving_dates + pd.Timedelta(days=1)
df["is_black_friday"] = df[DATE_COL].isin(black_friday_dates).astype(int)

# Christmas Eve / New Year's Eve
df["is_christmas_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 24)
).astype(int)

df["is_new_year_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 31)
).astype(int)

# -----------------------------------------------------------------------------
# HOLIDAY PROXIMITY FEATURES
# -----------------------------------------------------------------------------
# "near holiday" windows often help more than holiday-day itself
for lag in [1, 2, 3, 7]:
    df[f"is_pre_holiday_{lag}"] = 0
    df[f"is_post_holiday_{lag}"] = 0

for h in holiday_dates:
    for lag in [1, 2, 3, 7]:
        df.loc[df[DATE_COL] == h - pd.Timedelta(days=lag), f"is_pre_holiday_{lag}"] = 1
        df.loc[df[DATE_COL] == h + pd.Timedelta(days=lag), f"is_post_holiday_{lag}"] = 1

# -----------------------------------------------------------------------------
# LONG WEEKEND / BRIDGE-DAY FEATURES
# -----------------------------------------------------------------------------
# Friday before holiday Monday, Monday after holiday weekend, etc.
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)

df["is_bridge_day"] = 0
holiday_set = set(holiday_dates)

for i, d in enumerate(df[DATE_COL]):
    prev_day = d - pd.Timedelta(days=1)
    next_day = d + pd.Timedelta(days=1)
    # workday between holiday and weekend
    if (prev_day in holiday_set and d.dayofweek == 4) or (next_day in holiday_set and d.dayofweek == 0):
        df.at[i, "is_bridge_day"] = 1
# USE THE EXOGENOUS COLUMNS YOU CREATED
#EXOG_COLS = ["day_of_week", "doy", "is_thanksgiving", "is_christmas","is_weekend"]

EXOG_COLS = [
    # O tempo flui de forma cíclica (sem saltos de 31 para 1)
    "dow_sin", "dow_cos", 
    "month_sin", "month_cos",
    
    # Flags principais (removidas as pré e pós feriados a não ser que sejam precisas)
    "is_weekend",
    "is_holiday", 
    "is_christmas",
    "is_black_friday",
    
    # Evitar colocar dezenas de feriados e lags se o dataset for pequeno
]

76011


In [6]:
df

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_post_holiday_1,is_pre_holiday_2,is_post_holiday_2,is_pre_holiday_3,is_post_holiday_3,is_pre_holiday_7,is_post_holiday_7,is_bridge_day
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021-01-23,156,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021-01-23,260,6,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021-01-23,278,26,sour cream,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
4,2021-01-23,314,4,cottage chs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76006,2023-02-22,959943,3,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76007,2023-02-22,961844,6,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76008,2023-02-22,962444,13,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76009,2023-02-22,983754,4,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df["item_id"].unique()

array([    27,    156,    260,    278,    314,    324,    342,    347,
          348,    360,    779,    903,    904,    980,   1260,   1325,
         1327,  16961,  18605,  18607,  19615,  20082,  20253,  20605,
        20965,  21161,  26164,  26220,  26304,  26636,  26792,  26862,
        26910,  26924,  29629,  30753,  32180,  32239,  47364,  48371,
        48372,  48373,  48375,  53667,  97739, 113632, 113775, 113887,
       124581, 129834, 132356, 132357, 132359, 132363, 237588, 241839,
       256207, 270075, 288716, 501132, 501469, 501471, 501490, 501711,
       505414, 530684, 535948, 536321, 536433, 538923, 545598, 545604,
       545621, 587317, 606214, 606230, 606257, 606279, 606290, 606300,
       606365, 606383, 606402, 606415, 613934, 613935, 628129, 640017,
       640028, 936601, 936693, 937490, 945892, 945964, 959943, 961844,
       962444, 962469, 983754, 988016])

# Graph Construction setup

In [8]:
WINDOW_SIZE = 15
STEP_SIZE = 7
SIMILARITY_METHOD = "kendall"
SIMILARITY_THRESHOLD = 0.7

In [ ]:
import pickle

# Caminho para o ficheiro exportado (ajuste se guardou noutra pasta)
pkl_path = 'DynamicGraphPkls/dynamic_graphs_output_kendall_0.8_Window7_Step1.pkl'

print(f"A carregar grafos e features pré-computados de {pkl_path}...")
with open(pkl_path, 'rb') as f:
    graph_data = pickle.load(f)

# Extrair todas as variáveis de volta para o ambiente do notebook
graphs = graph_data['graphs']
sim_dfs = graph_data['sim_dfs']
df_pivots = graph_data['df_pivots']
window_info = graph_data['window_info']
dynamic_graph_features = graph_data['dynamic_graph_features']

print(f"Sucesso! Foram carregadas as informações de {len(graphs)} janelas temporais.")
print(f"Dimensão das features do nó na janela 1: {dynamic_graph_features[0].shape}")

A carregar grafos e features pré-computados de dynamic_graphs_output.pkl...
Sucesso! Foram carregadas as informações de 107 janelas temporais.
Dimensão das features do nó na janela 1: torch.Size([100, 8])


# GNN Embedding Strategy

In [10]:
from gnn_embedding_strategy import LateFusionGCNLSTM, GCN_LSTM_InitState, GCN_LSTM_ConcatPerStep

seq_length = 30
lookback_window = 7 
N_LAYERS = 1
LSTM_BATCH_SIZE = 32
LSTM_HIDDEN_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GCN_INPUT_DIM = 8  # Because compute_window_node_features yields 8 features per node
GCN_HIDDEN_DIM = 32
GCN_OUTPUT_DIM = 16

gcn_model = GCN(nfeat=GCN_INPUT_DIM, nhid=GCN_HIDDEN_DIM, nclass=GCN_OUTPUT_DIM, dropout=0.2)
embedding_strategy = "concat_per_step"  # Options: "late_fusion", "init_state", "concat_per_step"

# Using the Graph Node Embeddings as exogenous variables injected at each timestep of the LSTM
if embedding_strategy == "concat_per_step":
    gcn_lstm_model = GCN_LSTM_ConcatPerStep(
        gcn_model=gcn_model,
        lstm_input_size=1 + len(EXOG_COLS),  # Univariate (1) + calendar exogenous counts
        lstm_hidden_size=LSTM_HIDDEN_SIZE,
        lstm_num_layers=N_LAYERS,
        gcn_embed_dim=GCN_OUTPUT_DIM,        # Plugs the graph sizes directly in as features
        horizon=1,
        dropout=0.0
    ).to(device)

if embedding_strategy == "init_state":
    gcn_lstm_model = GCN_LSTM_InitState(
        gcn_model=gcn_model,
        lstm_input_size=1 + len(EXOG_COLS),  # Univariate (1) + calendar exogenous counts
        lstm_hidden_size=LSTM_HIDDEN_SIZE,
        lstm_num_layers=N_LAYERS,
        gcn_embed_dim=GCN_OUTPUT_DIM,        # Plugs the graph sizes directly in as features
        horizon=1,
        dropout=0.0
    ).to(device)


In [ ]:
from torch.utils.data import Dataset, DataLoader
import networkx as nx
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from gcn_lstm_dataset import GraphBatchSampler, SingleItemGraphDataset, get_adj_matrix


def compute_window_node_features(df_pivot: pd.DataFrame):
    num_items = df_pivot.shape[1]
    window_size = df_pivot.shape[0]
    features = []
    
    for j in range(num_items):
        item_ts = df_pivot.iloc[:, j].values
        last_demand = item_ts[-1] if window_size > 0 else 0
        mean7 = np.mean(item_ts[-7:]) if window_size >= 7 else np.mean(item_ts)
        mean28 = np.mean(item_ts[-28:]) if window_size >= 28 else np.mean(item_ts)
        std28 = np.std(item_ts[-28:]) if window_size >= 28 else np.std(item_ts)
        if window_size >= 28:
            zero_ratio28 = np.mean(item_ts[-28:] == 0)
            slope28 = np.polyfit(np.arange(28), item_ts[-28:], 1)[0]
            min_28 = np.min(item_ts[-28:])
            max_28 = np.max(item_ts[-28:])
        elif window_size > 1:
            zero_ratio28 = np.mean(item_ts == 0)
            slope28 = np.polyfit(np.arange(window_size), item_ts, 1)[0]
            min_28 = np.min(item_ts)
            max_28 = np.max(item_ts)
        else:
            zero_ratio28 = np.mean(item_ts == 0)
            slope28 = 0.0
            min_28 = item_ts[0] if window_size > 0 else 0
            max_28 = item_ts[0] if window_size > 0 else 0
            
        features.append([last_demand, mean7, mean28, std28, zero_ratio28, slope28, min_28, max_28])
    return np.array(features)

def gcn_lstm_dynamic_last_fusion(
    df,
    target,
    item_id,
    store_id,
    gcn_lstm_model,
    train_size,
    val_size,
    forecast_window,
    lookback_window,
    seq_length,
    batch_size,
    graphs_list,
    graph_features_list,
    window_info,
    exog_cols=None,
    num_epochs=100,
    lr=0.001,
    patience=10,
    horizon=1, 
    dropout=0.2,
    seed=42,
    loss_type="MSELoss",
    save_plot_path=None,
    device="cpu"
):
    start_train_time = time.time()
    test_start_idx = len(df) - forecast_window
    val_start_idx = test_start_idx - val_size
    train_start_idx = val_start_idx - train_size

    print(f"Calculated indices - Train Start: {train_start_idx}, Val Start: {val_start_idx}, Test Start: {test_start_idx}")
    if train_start_idx < 0:
        train_start_idx = 0
        print(f"Warning: Dataset shorter than requested split sizes. adjusting train_start to 0.")

    train_slice = slice(train_start_idx, val_start_idx)
    val_slice = slice(val_start_idx, test_start_idx)
    test_slice = slice(test_start_idx, None)
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # Extract Target
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    # Scale Target
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    # Extract Exogenous Variables
    if exog_cols and len(exog_cols) > 0:
        exog_train = df[exog_cols][train_slice].values
        exog_val = df[exog_cols][val_slice].values
        exog_test = df[exog_cols][test_slice].values
        exog_scaler = MinMaxScaler()
        exog_train_scaled = exog_scaler.fit_transform(exog_train)
        exog_val_scaled = exog_scaler.transform(exog_val)
        exog_test_scaled = exog_scaler.transform(exog_test)
    else:
        exog_train_scaled = None
        exog_val_scaled = None
        exog_test_scaled = None
         
    # Node index mapping
    g0 = graphs_list[0]
    node_list = list(g0.keys()) if isinstance(g0, dict) else list(g0.nodes())
    if item_id in node_list:
        target_node_idx = node_list.index(item_id)
    else:
        target_node_idx = 0 # fallback

    train_dates = df['date'][train_slice].values
    val_dates = df['date'][val_slice].values

    train_dataset = SingleItemGraphDataset(train_scaled, exog_train_scaled, seq_length, train_dates, window_info, target_node_idx)
    train_sampler = GraphBatchSampler(train_dataset, batch_size=batch_size, shuffle=False)
    use_pin_memory = torch.cuda.is_available()
    
    if len(train_dataset) == 0:
        print("Warning: Train dataset is empty.")
    train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, shuffle=False, pin_memory=use_pin_memory)
    
    val_dataset = SingleItemGraphDataset(val_scaled, exog_val_scaled, seq_length, val_dates, window_info, target_node_idx)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=use_pin_memory)
    
    if loss_type == 'MSELoss':
        criterion = nn.MSELoss(reduction='none')
    elif loss_type == 'L1Loss':
        criterion = nn.L1Loss(reduction='none')
    elif loss_type == 'HuberLoss':
        criterion = nn.HuberLoss(reduction='none')
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")
    
    model = gcn_lstm_model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    epochs_no_improve = 0
    best_model_state = None
    best_epoch = 0

    adj_tensors = []
    feat_tensors = []
    for g, feat in zip(graphs_list, graph_features_list):
        if isinstance(g, dict):
            adj = get_adj_matrix(g)
        else:
            adj = nx.adjacency_matrix(g).todense()
        adj_tensors.append(torch.tensor(adj, dtype=torch.float32).to(device))
        feat_tensors.append(feat.to(device).clone().detach().float())

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            ts_x = batch["ts_x"].to(device)
            y = batch["y"].to(device)
            graph_idx_batch = batch["graph_idx"].unique()
            target_node_batch = batch["target_node_idx"].to(device)
            
            optimizer.zero_grad()
            total_elements = len(ts_x)
            
            total_loss_tensor = 0.0
            
            for g_idx in graph_idx_batch:
                mask = (batch["graph_idx"] == g_idx)
                if not mask.any():
                    continue
                
                curr_ts_x = ts_x[mask]
                curr_y = y[mask]
                curr_target_nodes = target_node_batch[mask]

                g_i = g_idx.item()
                adj_tensor = adj_tensors[g_i]
                feats = feat_tensors[g_i]

                preds = model(curr_ts_x, feats, adj_tensor, curr_target_nodes)
                
                if preds.shape != curr_y.shape:
                    curr_y = curr_y.view_as(preds)
                
                loss_elements = criterion(preds, curr_y)
                total_loss_tensor = total_loss_tensor + loss_elements.sum()
                
            avg_loss = total_loss_tensor / total_elements
            avg_loss.backward()
            optimizer.step()
            
            if isinstance(total_loss_tensor, torch.Tensor):
                train_loss += total_loss_tensor.item()
            else:
                train_loss += total_loss_tensor

        if len(train_loader.dataset) > 0:
            train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                ts_x = batch["ts_x"].to(device)
                y = batch["y"].to(device)
                graph_idx_batch = batch["graph_idx"].unique()
                target_node_batch = batch["target_node_idx"].to(device)

                batch_loss_val = 0.0
                for g_idx in graph_idx_batch:
                    mask = (batch["graph_idx"] == g_idx)
                    if not mask.any():
                        continue
                    
                    curr_ts_x = ts_x[mask]
                    curr_y = y[mask]
                    curr_target_nodes = target_node_batch[mask]

                    g_i = g_idx.item()
                    adj_tensor = adj_tensors[g_i]
                    feats = feat_tensors[g_i]

                    preds = model(curr_ts_x, feats, adj_tensor, curr_target_nodes)
                    
                    if preds.shape != curr_y.shape:
                        curr_y = curr_y.view_as(preds)
                        
                    loss_elements = criterion(preds, curr_y)
                    batch_loss_val += loss_elements.sum().item()
                    
                val_loss += batch_loss_val
                
        if len(val_loader.dataset) > 0:
            val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_no_improve = 0
            best_model_state = model.state_dict()
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    train_time = time.time() - start_train_time

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    model.eval()
    start_inference_time = time.time()
    forecast = []
    
    # Get the graph configuration for inference (last available graph from validation phase)
    g_i = len(graphs_list) - 1
    adj_tensor = adj_tensors[g_i]
    feat_tensor = feat_tensors[g_i]

    target_node_tensor = torch.tensor([target_node_idx], dtype=torch.long).to(device)
    
    history_slice = slice(test_start_idx - lookback_window, test_start_idx)
    current_seq_unscaled = df[target][history_slice].values
    current_seq = scaler.transform(current_seq_unscaled.reshape(-1, 1)).flatten().tolist()
    
    if exog_cols and len(exog_cols) > 0:
        current_exog_seq_unscaled = df[exog_cols][history_slice].values
        current_exog_seq = exog_scaler.transform(current_exog_seq_unscaled).tolist()
        exog_test = df[exog_cols][test_slice].values
    else:
        current_exog_seq = []
        exog_test = []

    test = df[target][test_slice].values
    test_unscaled = test.copy()
    
    import os
    inf_log_dir = f'inference_logs/seed_{seed}'
    os.makedirs(inf_log_dir, exist_ok=True)
    inf_log_path = f'{inf_log_dir}/inference_item{item_id}.csv'

    with torch.no_grad():
        for step in range(forecast_window):
            if exog_cols and len(exog_cols) > 0:
                current_seq_arr = np.array(current_seq).reshape(-1, 1)
                current_exog_arr = np.array(current_exog_seq)
                x_np = np.column_stack([current_seq_arr, current_exog_arr])
            else:
                x_np = np.array(current_seq).reshape(-1, 1)

            x = torch.FloatTensor(x_np).unsqueeze(0).to(device)

            pred = model(x, feat_tensor, adj_tensor, target_node_tensor).cpu().numpy()[0, 0]
            pred_unscaled = scaler.inverse_transform([[pred]])[0, 0]
            forecast.append(pred_unscaled) # Store unscaled
            
            # Slide window
            current_seq = current_seq[1:] + [pred]
            if exog_cols and len(exog_cols) > 0 and step + 1 < forecast_window:
                next_exog_scaled = exog_test_scaled[step + 1].copy()
                current_exog_seq = current_exog_seq[1:] + [next_exog_scaled.tolist()]
                
    forecast_unscaled = np.array(forecast).flatten()
    print(f"Forecasted values (Unscaled) {forecast_unscaled[:5]} ...")
    infer_time = time.time() - start_inference_time
    
    rmse = np.sqrt(mean_squared_error(test_unscaled, forecast_unscaled))
    mae = mean_absolute_error(test_unscaled, forecast_unscaled)
    bias = np.mean(forecast_unscaled - test_unscaled)
    score = 0.5 * rmse + 0.25 * mae + 0.25 * abs(bias)
    def POCID(y_test, y_pred):
        diff_original = y_test[1:] - y_test[:-1]
        diff_pred = y_pred[1:] - y_pred[:-1]
        is_positive = (diff_original * diff_pred) > 0
        return is_positive.sum() / len(is_positive) if len(is_positive) > 0 else 0.0

    pocid = POCID(test_unscaled, forecast_unscaled)
    if save_plot_path:
        from model_utils.plots import plot_results
        train_index = df['date'][train_slice].values
        val_index = df['date'][val_slice].values
        test_index = df['date'][test_slice].values
        
        plot_results(train, val, test_unscaled, forecast_unscaled, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=rmse, pocid=pocid)
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, Bias: {bias:.4f}")
    
    return rmse, mae, train_time, infer_time, best_epoch


In [12]:
# Grid Search Parameters
seeds = [2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
dropout = 0.0
EPOCHS = 150  # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

# Filter for specific products if needed
target_products = [27]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)


In [ ]:
# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 154
forecast_horizon = 152

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    df_product = df_product.reset_index(drop=True)
                else:
                    df_product = df_product.reset_index()

            df_product = df_product.reset_index(drop=True)
            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True)
            
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            
            rmse, mae, train_time, infer_time, best_epoch = gcn_lstm_dynamic_last_fusion(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                gcn_lstm_model=gcn_lstm_model,
                train_size=train_size,
                val_size=val_size,
                forecast_window=forecast_horizon,
                lookback_window=lookback_window,
                seq_length=seq_length,
                batch_size=batch_size,
                graphs_list=graphs,
                graph_features_list=dynamic_graph_features,
                window_info=window_info,
                exog_cols=EXOG_COLS,
                num_epochs=EPOCHS,
                lr=LEARNING_RATE,
                dropout=dropout,
                patience=1000,
                horizon=1,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename,
                device=device
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        
# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)


Starting Grid Search with 1 seeds, 1 loss functions and 1 products...

--- Processing Seed: 2024 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=2024, Loss=MSELoss, Item=27, Store=6269
Calculated indices - Train Start: 0, Val Start: 455, Test Start: 609
Epoch 10/150 | Train Loss: 0.031379 | Val Loss: 0.061070
Epoch 20/150 | Train Loss: 0.028940 | Val Loss: 0.036573
Epoch 30/150 | Train Loss: 0.027901 | Val Loss: 0.035232
Epoch 40/150 | Train Loss: 0.026533 | Val Loss: 0.034802
Epoch 50/150 | Train Loss: 0.025094 | Val Loss: 0.033065
Epoch 60/150 | Train Loss: 0.023769 | Val Loss: 0.032891
Epoch 70/150 | Train Loss: 0.023024 | Val Loss: 0.031807
Epoch 80/150 | Train Loss: 0.022276 | Val Loss: 0.024638
Epoch 90/150 | Train Loss: 0.020643 | Val Loss: 0.038753
Epoch 100/150 | Train Loss: 0.020967 | Val Loss: 0.041198
Epoch 110/150 | Train Loss: 0.019772 | Val Loss: 0.034430
Epoch 120/150 | Train Loss: 0.018809 | Val Loss: 0.029593
Epoch 130/150 | Train Loss: 0.019200 | Val Loss: 0

TypeError: Invalid value '20.93673723936081' for dtype 'int32'